# Task 1: Dataset Exploration & Preprocessing
## SWYNEX Technologies — AI/ML Internship

**Dataset:** Titanic passenger survival dataset (public CSV mirror)  
**Source:** [DataScienceDojo Titanic CSV](https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv)  
**Goal:** Explore the dataset, inspect data quality, handle missing values, encode categorical variables, and prepare a train/test feature pipeline.

> Run this notebook from top to bottom in Google Colab or Jupyter. The first data-loading cell downloads the CSV from the public source. Keep the raw data unchanged and perform transformations on a copy.


## 1. Import libraries

This notebook uses common Python data-science libraries for tabular analysis and visualization.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


## 2. Load the dataset

The dataset is a public copy of the commonly used Titanic passenger training data. If the source URL changes, download the CSV and upload it to Colab, then replace the URL with the uploaded file path.


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

df_raw = pd.read_csv(DATA_URL)
df = df_raw.copy()

print("Dataset loaded successfully")
print("Shape:", df.shape)
display(df.head())


## 3. Initial exploration

Review dimensions, column names, data types, summary statistics, and categorical values before deciding how to preprocess.


In [ ]:
print("Rows and columns:", df.shape)
print("\\nColumn names:")
print(df.columns.tolist())

print("\\nData types and non-null counts:")
df.info()

print("\\nNumerical summary:")
display(df.describe().T)

print("\\nCategorical summary:")
display(df.describe(include="object").T)


## 4. Data quality checks

Check missing values, duplicate records, and repeated passenger IDs. Do not remove rows automatically without understanding what they represent.


In [ ]:
missing_count = df.isna().sum().sort_values(ascending=False)
missing_percent = (df.isna().mean() * 100).sort_values(ascending=False)

missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_percent.round(2)
})
display(missing_summary)

print("Exact duplicate rows:", df.duplicated().sum())
if "PassengerId" in df.columns:
    print("Duplicate PassengerId values:", df["PassengerId"].duplicated().sum())


## 5. Exploratory data analysis (EDA)

### Target distribution and passenger characteristics


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.countplot(data=df, x="Survived", ax=axes[0])
axes[0].set_title("Survival Count")
axes[0].set_xticks([0, 1], ["Did not survive", "Survived"])

sns.countplot(data=df, x="Pclass", ax=axes[1])
axes[1].set_title("Passenger Class")

sns.countplot(data=df, x="Sex", hue="Survived", ax=axes[2])
axes[2].set_title("Survival by Sex")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(data=df, x="Age", kde=True, bins=30, ax=axes[0])
axes[0].set_title("Age Distribution")

sns.histplot(data=df, x="Fare", kde=True, bins=30, ax=axes[1])
axes[1].set_title("Fare Distribution")

plt.tight_layout()
plt.show()


In [ ]:
survival_by_class = df.groupby("Pclass", observed=False)["Survived"].mean().rename("survival_rate")
survival_by_sex = df.groupby("Sex", observed=False)["Survived"].mean().rename("survival_rate")
survival_by_embarked = df.groupby("Embarked", observed=False)["Survived"].mean().rename("survival_rate")

print("Survival rate by passenger class:")
display(survival_by_class.to_frame().style.format("{:.1%}"))

print("Survival rate by sex:")
display(survival_by_sex.to_frame().style.format("{:.1%}"))

print("Survival rate by port of embarkation:")
display(survival_by_embarked.to_frame().style.format("{:.1%}"))


## 6. Handle missing values and engineer useful features

Preprocessing decisions:
- `Age`: fill missing values with the median (robust to skew and outliers).
- `Embarked`: fill missing values with the most frequent category.
- `Cabin`: very sparse; create a `CabinKnown` indicator, then remove the raw cabin text.
- `Name`, `Ticket`, and `PassengerId`: remove from model features because they are identifiers or high-cardinality text for this baseline task.
- Create `FamilySize` and `IsAlone` from family counts.

For the modeling pipeline below, imputers and scalers are fitted only on the training split to avoid data leakage.


In [ ]:
df_clean = df.copy()

# Basic duplicate removal (if any exact duplicates exist)
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

# Cabin is mostly missing: retain whether cabin information was recorded
df_clean["CabinKnown"] = df_clean["Cabin"].notna().astype(int)
df_clean = df_clean.drop(columns=["Cabin"])

# Simple, interpretable feature engineering
df_clean["FamilySize"] = df_clean["SibSp"] + df_clean["Parch"] + 1
df_clean["IsAlone"] = (df_clean["FamilySize"] == 1).astype(int)

# Remove identifier / raw text columns for this baseline feature set
drop_cols = [c for c in ["PassengerId", "Name", "Ticket"] if c in df_clean.columns]
df_clean = df_clean.drop(columns=drop_cols)

print("Shape after basic cleaning and feature engineering:", df_clean.shape)
display(df_clean.head())


## 7. Prepare features and target

Separate the target (`Survived`) from input features. Then split the data into training and testing sets before fitting preprocessing steps.


In [ ]:
target = "Survived"
X = df_clean.drop(columns=[target])
y = df_clean[target].astype(int)

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape, y_train.shape)
print("Testing set:", X_test.shape, y_test.shape)


## 8. Encode categorical variables and scale numerical features

Use:
- Median imputation + standard scaling for numeric columns.
- Most-frequent imputation + one-hot encoding for categorical columns.

`handle_unknown="ignore"` allows the pipeline to handle categories that appear in the test data but not the training data.


In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

X_train_processed_df = pd.DataFrame(
    X_train_processed, columns=feature_names, index=X_train.index
)
X_test_processed_df = pd.DataFrame(
    X_test_processed, columns=feature_names, index=X_test.index
)

print("Processed training features:", X_train_processed_df.shape)
print("Processed testing features:", X_test_processed_df.shape)
display(X_train_processed_df.head())


## 9. Verify preprocessing

Confirm that the transformed feature matrices contain no missing values and that categorical variables have been converted to numeric columns.


In [ ]:
print("Missing values in processed train:", np.isnan(X_train_processed).sum())
print("Missing values in processed test:", np.isnan(X_test_processed).sum())
print("All processed features numeric:", np.issubdtype(X_train_processed.dtype, np.number))

print("\\nProcessed feature names:")
print(list(feature_names))


## 10. Export cleaned data and processed features (optional)

These files are generated when the notebook is run. The cleaned table is useful for inspection; the train/test processed files are ready to be used as input to a model. The target labels are saved separately.


In [ ]:
df_clean.to_csv("titanic_cleaned.csv", index=False)
X_train_processed_df.to_csv("X_train_processed.csv", index=False)
X_test_processed_df.to_csv("X_test_processed.csv", index=False)
y_train.to_frame(name=target).to_csv("y_train.csv", index=False)
y_test.to_frame(name=target).to_csv("y_test.csv", index=False)

print("Exported cleaned dataset and processed train/test files.")


## 11. Conclusion

In this task, we:
1. Loaded and inspected a public passenger dataset.
2. Summarized numerical and categorical columns.
3. Checked missing values and duplicate records.
4. Performed EDA with plots and grouped survival rates.
5. Handled sparse cabin information and engineered family-related features.
6. Imputed missing values, encoded categories, and scaled numerical features.
7. Prepared training and testing feature matrices without fitting preprocessing on the test set.

**Note:** This notebook focuses on dataset exploration and preprocessing. It does not claim to train or evaluate a prediction model.
